# Recollida de dades

Aquest notebook genera la mostra inicial del treball: 1000 colors RGB aleatoris, el seu chroma i les imatges PNG que despres passarem als models.

La logica important esta a `scripts/utils.py`. Aqui intento deixar nomes la configuracio i les crides principals, perque si no el notebook es fa impossible de seguir.

## Configuracio

Fixem els parametres de l'experiment. La seed es important per poder regenerar exactament la mateixa mostra en un altre ordinador.

In [ ]:
from pathlib import Path
import sys

N_COLORS = 1000
SEED = 23
IMAGE_SIZE = 100
NEAR_DISTANCE = 30

# Aquesta part crida l'API i pot gastar diners. Ho deixo apagat per defecte.
RUN_MODEL_QUERIES = False
MODEL_NAMES = ["gpt-4o-mini"]
MODEL_TEMPERATURE = 0.2
MODEL_MAX_IMAGES = None  # posa un numero petit, per exemple 5, si vols fer una prova rapida

# Canvia a True nomes si vols tornar a generar la mostra des de zero.
REGENERATE_SAMPLE = True

# Casella de seguretat: si es posa a True, esborra CSVs, imatges PNG i logs generats.
DELETE_EXISTING_SAMPLE = False

base_dir = Path("recollida-dades") if Path("recollida-dades").exists() else Path(".")
csv_dir = base_dir / "csv"
images_dir = base_dir / "images"
tmp_dir = base_dir / "tmp"
logs_dir = base_dir / "logs"
scripts_dir = base_dir / "scripts"

for folder in [csv_dir, images_dir, tmp_dir, logs_dir]:
    folder.mkdir(parents=True, exist_ok=True)

if str(scripts_dir) not in sys.path:
    sys.path.append(str(scripts_dir))

print("Carpeta base:", base_dir)


## Carrega de scripts

Importem les funcions del script. Si alguna no existeix, parem aqui, perque probablement el notebook no esta trobant be `scripts/utils.py`.

In [ ]:
import importlib
import pandas as pd
from IPython.display import Image as DisplayImage, display

import utils
importlib.reload(utils)

build_final_sample = utils.build_final_sample
collect_model_outputs = utils.collect_model_outputs
delete_sample_outputs = utils.delete_sample_outputs
delete_png_files = utils.delete_png_files
generate_images_from_sample = utils.generate_images_from_sample
generate_unique_colors = utils.generate_unique_colors
save_csv = utils.save_csv
save_rgb_sample_grid = utils.save_rgb_sample_grid
save_rgb_sample_map = utils.save_rgb_sample_map
save_chroma_distribution = utils.save_chroma_distribution
write_log = utils.write_log

required_functions = [
    "generate_unique_colors",
    "generate_images_from_sample",
    "save_rgb_sample_grid",
    "save_rgb_sample_map",
    "save_chroma_distribution",
    "delete_sample_outputs",
    "delete_png_files",
    "save_csv",
    "write_log",
]
missing = [name for name in required_functions if name not in globals()]

if missing:
    raise RuntimeError(f"Error: scripts no cargados correctamente: {missing}")

if DELETE_EXISTING_SAMPLE:
    removed = delete_sample_outputs(csv_dir=csv_dir, images_dir=images_dir, logs_dir=logs_dir)
    print(f"Fitxers esborrats: {len(removed)}")

write_log("Notebook de recollida inicialitzat", logs_dir / "pipeline.log")


## Generacio o carrega de la mostra de colors

Si `input_image_sample.csv` ja existeix, el carreguem i no el tornem a generar. Aixi evitem canviar la mostra sense voler.

Per regenerar-ho tot, cal posar `DELETE_EXISTING_SAMPLE = True` i `REGENERATE_SAMPLE = True` a la configuracio. Ho deixo aixi una mica explicit per no liar-la per accident.


In [ ]:
input_path = csv_dir / "input_image_sample.csv"

if input_path.exists() and not REGENERATE_SAMPLE:
    input_sample = pd.read_csv(input_path)
    print(f"Mostra carregada: {input_path}")
else:
    input_sample = generate_unique_colors(n=N_COLORS, seed=SEED)
    save_csv(input_sample, input_path)
    write_log(f"Mostra generada: {input_path}", logs_dir / "pipeline.log")
    print(f"Mostra generada: {input_path}")

input_sample.head()


## Mapa de color i comprovacio rapida de biaix

Aqui genero tres visualitzacions. No son una prova matematica perfecta, pero ajuden bastant a veure si la mostra fa coses rares.

- **Graella de colors**: mostra els 1000 colors exactes ordenats per to. Serveix per mirar la mostra a ull, no per demostrar uniformitat.
- **Distribucio per canal RGB**: son tres histogrames, un per R, G i B. Com que els RGB surten aleatoris, les barres haurien de quedar mes o menys equilibrades, pero no identiques.
- **Hue vs saturation**: posa el to a l'eix X i la saturacio a l'eix Y. Els punts de baix son colors mes grisos. Important: una mostra uniforme en RGB no te per que ser uniforme en HSV.
- **R vs G, R vs B i G vs B**: son projeccions del cub RGB. El que volem veure es que els punts omplen el quadrat sense forats grans ni patrons massa clars.
- **max RGB vs min RGB**: compara el canal mes alt amb el mes baix de cada color. La forma triangular es normal, perque el minim mai pot ser mes gran que el maxim.
- **Distribucio del chroma**: chroma baix vol dir color proper al gris, i chroma alt vol dir color mes viu o saturat. Aquesta distribucio tampoc ha de ser plana obligatoriament, perque depen de com RGB es transforma a Lab.

Els punts es pinten amb el seu color real, sense contorns blancs o negres, per no confondre la visualitzacio. Si surt algun punt quasi negre, es perque aquell color realment es fosc.


In [ ]:
color_grid_path = tmp_dir / "rgb_sample_grid.png"
color_map_path = tmp_dir / "rgb_sample_map.png"
chroma_plot_path = tmp_dir / "rgb_chroma_distribution.png"

# Les visualitzacions son barates de generar, aixi que les fem sempre a partir del CSV actual.
# D'aquesta manera no ens quedem mirant una imatge antiga del notebook.
save_rgb_sample_grid(input_sample, color_grid_path)
save_rgb_sample_map(input_sample, color_map_path)
save_chroma_distribution(input_sample, chroma_plot_path)
write_log(f"Visualitzacions RGB generades a tmp: {color_grid_path}, {color_map_path}, {chroma_plot_path}", logs_dir / "pipeline.log")

print(f"Graella generada: {color_grid_path}")
print(f"Mapa de diagnosi generat: {color_map_path}")
print(f"Grafic de chroma generat: {chroma_plot_path}")

print("Graella de les 1000 mostres:")
display(DisplayImage(filename=str(color_grid_path)))

print("Mapa de diagnosi de distribucio:")
display(DisplayImage(filename=str(color_map_path)))

print("Distribucio del chroma:")
display(DisplayImage(filename=str(chroma_plot_path)))

input_sample[["r", "g", "b", "chroma"]].describe()


## Generacio de les imatges

Les imatges individuals tambe es reutilitzen si ja existeixen. Si falten o si activem `REGENERATE_SAMPLE`, primer esborrem els PNG antics de `images/` i despres es tornen a crear.

Cada imatge es de `IMAGE_SIZE x IMAGE_SIZE` pixels. La composicio es: 60% de pixels exactament del color objectiu, 20% de pixels amb colors propers pero aleatoris entre ells, i el 20% restant amb colors aleatoris de tot l'espectre RGB. Les guardem en PNG per no modificar el color amb compressio JPEG.


In [ ]:
expected_images = {name for name in input_sample["image_name"]}
existing_images = {path.name for path in images_dir.glob("*.png")}
missing_images = expected_images - existing_images

if missing_images or REGENERATE_SAMPLE:
    removed_images = delete_png_files(images_dir)
    if removed_images:
        print(f"Imatges antigues esborrades: {len(removed_images)}")

    image_paths = generate_images_from_sample(
        input_sample,
        output_dir=images_dir,
        size=IMAGE_SIZE,
        near_distance=NEAR_DISTANCE,
        seed=SEED,
    )
    write_log(f"Imatges antigues esborrades: {len(removed_images)}", logs_dir / "pipeline.log")
    write_log(f"Imatges generades: {len(image_paths)}", logs_dir / "pipeline.log")
    print(f"Imatges generades: {len(image_paths)}")
else:
    image_paths = sorted(images_dir.glob("*.png"))
    print(f"Imatges ja existents: {len(image_paths)}")

len(image_paths), image_paths[:3]


In [ ]:
from pathlib import Path
images_path = Path("images")
num_files = len(list(images_path.glob("*.png")))
print('Conteig imatges observades:',num_files)

## Consulta als models

Aquesta part ja queda preparada per executar el benchmark amb models de visio. La clau no s'ha d'escriure mai al notebook: s'ha de posar en un fitxer `.env` local amb `OPENAI_API_KEY=...`. Aquest fitxer esta ignorat per Git.

Per evitar ensurts, `RUN_MODEL_QUERIES` esta a `False` per defecte. Quan ho vulgui executar de veritat, el canvio a `True`. Si falla a mig cami, els resultats es van guardant a `csv/outputmodel_image_sample.csv`, i la propera execucio salta les parelles imatge/model que ja existeixen.


In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

output_path = csv_dir / "outputmodel_image_sample.csv"
model_log_path = logs_dir / "model_calls.log"

if RUN_MODEL_QUERIES:
    if not os.getenv("OPENAI_API_KEY"):
        raise RuntimeError("Falta OPENAI_API_KEY. Crea un fitxer .env local amb la clau abans d'executar models.")

    client = OpenAI()
    model_outputs = collect_model_outputs(
        client=client,
        image_paths=image_paths,
        models=MODEL_NAMES,
        temperature=MODEL_TEMPERATURE,
        output_path=output_path,
        log_file=model_log_path,
        max_images=MODEL_MAX_IMAGES,
    )

    print(f"Resultats de models guardats: {output_path}")
    print(f"Files totals: {len(model_outputs)}")
    display(model_outputs.tail())
else:
    if output_path.exists():
        model_outputs = pd.read_csv(output_path)
        print(f"Models no executats ara. Carrego resultats existents: {len(model_outputs)} files")
    else:
        model_outputs = pd.DataFrame()
        print("Models no executats. Posa RUN_MODEL_QUERIES = True quan vulguis cridar l'API.")


## Dataset final per analisi

Quan existeixi el CSV de sortida dels models, unim l'entrada i la sortida i calculem l'error cromatic en RGB. Aquest fitxer sera el que carregarem a R. Si encara no hem executat models, simplement avisa i no fa res.


In [ ]:
output_path = csv_dir / "outputmodel_image_sample.csv"
final_path = csv_dir / "sample-colors.csv"

if output_path.exists():
    model_outputs = pd.read_csv(output_path)
    final_sample = build_final_sample(input_sample, model_outputs)
    save_csv(final_sample, final_path)
    write_log(f"Dataset final guardat: {final_path} files={len(final_sample)}", logs_dir / "pipeline.log")
    final_sample.head()
else:
    print("Encara no existeix outputmodel_image_sample.csv. Primer cal executar la cel.la de models.")


## Resultats generats

Comprovem que tenim el CSV d'entrada i les imatges. Si aqui surt 1000, la part base de recollida ja esta feta.

In [ ]:
print("CSV generats:", sorted(path.name for path in csv_dir.glob("*.csv")))
print("Nombre d'imatges:", len(list(images_dir.glob("*.png"))))
print("Fitxers temporals:", sorted(path.name for path in tmp_dir.glob("*.png")))
print("Logs:", sorted(path.name for path in logs_dir.iterdir()))